# Customer Churn — Engineering Pipeline

**Purpose:** Reproducible, package-driven pipeline run. No exploratory code lives here.

For EDA, visualizations, and hypothesis testing see `Customer_Churn_Project.ipynb`.

---

| Step | Module |
|------|--------|
| Load raw data | `customer_churn_ml.data.loader` |
| Integrity check | `customer_churn_ml.utils` |
| Preprocess | `customer_churn_ml.preprocessing.preprocessor` |
| Split & scale | `customer_churn_ml.preprocessing.scalar` |
| Train & evaluate | `customer_churn_ml.training.trainer` |
| Interpret | `customer_churn_ml.interpret.feature_importance` |
| Artifacts | saved to `model/` via config |

## 0. Setup

In [ ]:
# If the package is not installed as editable, uncomment the line below.
# import sys; sys.path.insert(0, str(__import__('pathlib').Path().resolve().parent / 'src'))

import warnings

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 120

# ── Package imports (all logic lives here) ────────────────────────────────
from customer_churn_ml import (
    build_comparison_table,
    check_data_integrity,
    fit_scale_train_test,
    get_feature_importance_df,
    load_raw_data,
    plot_feature_importance,
    plot_roc_curves,
    preprocess_dataframe,
    train_models,
)
from customer_churn_ml.config import PATHS, RANDOM_STATE, TEST_SIZE
from customer_churn_ml.constants import TARGET_ENCODED_COL

print("Package loaded successfully.")
print(f"  random_state : {RANDOM_STATE}")
print(f"  test_size    : {TEST_SIZE}")
print(f"  model dir    : {PATHS.model_dir}")

## 1. Load Raw Data

In [ ]:
df_raw = load_raw_data()

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

## 2. Data Integrity Check

In [ ]:
report = check_data_integrity(df_raw)

print(f"Shape           : {report['shape']}")
print(f"Duplicate rows  : {report['duplicate_rows']}")
print(f"Duplicate IDs   : {report['duplicate_ids']}")

missing = {col: n for col, n in report["missing_values"].items() if n > 0}
if missing:
    print(f"Columns with missing values: {missing}")
else:
    print("Missing values  : none")

assert report["duplicate_rows"] == 0, "WARNING: duplicate rows detected — review before proceeding."
assert report["duplicate_ids"] == 0, "WARNING: duplicate customer IDs detected."

## 3. Preprocessing

In [ ]:
df_clean, preprocessor = preprocess_dataframe(df_raw, save_processed_data=True)

# Save the fitted preprocessor so predict.py can reuse it
preprocessor.save()

print(f"Clean shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"Target balance:\n{df_clean[TARGET_ENCODED_COL].value_counts().to_string()}")
df_clean.head(3)

## 4. Train / Test Split & Scaling

In [ ]:
X = df_clean.drop(columns=[TARGET_ENCODED_COL])
y = df_clean[TARGET_ENCODED_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Scaler is fit on train only — no data leakage into test
X_train, X_test, scaler = fit_scale_train_test(X_train, X_test)

print(f"Train : {X_train.shape[0]:,} rows  |  Test : {X_test.shape[0]:,} rows")
print(f"Features : {X_train.shape[1]}")
print(f"Churn rate — train: {y_train.mean():.1%}  |  test: {y_test.mean():.1%}")

## 5. Model Training

In [ ]:
outcome = train_models(
    X_train,
    y_train,
    X_test,
    y_test,
    save_artifacts=True,
)

print(f"Best model : {outcome.best_model_name}")
print(f"Saved to   : {PATHS.model_file}")

## 6. Evaluation

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────
comparison = pd.DataFrame(build_comparison_table(outcome.results))
print("Model comparison (sorted by ROC-AUC):")
display(comparison.style.format({"Accuracy": "{:.4f}", "ROC_AUC": "{:.4f}"}))

In [ ]:
# ── Per-model classification reports ─────────────────────────────────────
for model_name, metrics in outcome.results.items():
    print("=" * 55)
    print(f"  {model_name}")
    print("=" * 55)
    print(f"  Accuracy : {metrics['accuracy']:.4f}")
    print(f"  ROC-AUC  : {metrics['roc_auc']:.4f}")
    print()
    print(metrics["classification_report"])

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
plot_roc_curves(outcome.results, y_test, ax=ax)
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importance_df = get_feature_importance_df(
    outcome.best_model,
    outcome.feature_names,
    top_n=20,
)

fig, ax = plt.subplots(figsize=(8, 6))
plot_feature_importance(
    importance_df,
    title=f"Top 20 Features — {outcome.best_model_name}",
    ax=ax,
)
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 8. Artifact Summary

In [ ]:
artifacts = {
    "Cleaned data": PATHS.cleaned_data_file,
    "Scaler": PATHS.scaler_file,
    "Model": PATHS.model_file,
    "Feature names": PATHS.feature_names_file,
    "Preprocessor": PATHS.model_dir / "preprocessor.pkl",
}

print(f"{'Artifact':<20} {'Exists':<8} Path")
print("-" * 72)
for label, path in artifacts.items():
    exists = "✓" if path.exists() else "✗ MISSING"
    print(f"{label:<20} {exists:<8} {path}")